# 01 — Leakage study on BreakHis (run in Google Colab)

**Question:** how much does accuracy drop when we split by *patient* instead of by *image*?

Before you start: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
Results are saved to your Google Drive, so a disconnect doesn't lose finished runs — just re-run and it skips what's done.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/breakhis-results'

name, memory.total [MiB]
Tesla T4, 15360 MiB
Mounted at /content/drive


## 1. Get the code

In [2]:
!git clone https://github.com/amubakekengbam/breakhis-reliability.git /content/breakhis-reliability
%cd /content/breakhis-reliability
!pip install -q -e .

Cloning into '/content/breakhis-reliability'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 24 (delta 0), reused 24 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 11.49 KiB | 3.83 MiB/s, done.
/content/breakhis-reliability
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for breakhis-reliability (pyproject.toml) ... done


## 2. Download BreakHis (~1.6 GB, a few minutes)
From the official UFPR page. The authors ask you to fill in a short form after downloading and to cite
Spanhol et al. (2016) in any publication. Non-commercial research use only.

The archive is kept on Drive so you only download it once.

In [ ]:
import os
ARCHIVE = '/content/drive/MyDrive/datasets/BreaKHis_v1.tar.gz'
os.makedirs(os.path.dirname(ARCHIVE), exist_ok=True)
if not os.path.exists(ARCHIVE):
    !wget -q --show-progress -O "$ARCHIVE" http://www.inf.ufpr.br/vri/databases/BreaKHis_v1.tar.gz
if not os.path.exists('/content/BreaKHis_v1'):
    !tar -xzf "$ARCHIVE" -C /content/
DATA_DIR = '/content/BreaKHis_v1'

ontent/drive/MyDriv   0%[                    ]  19.51M  5.05MB/s    eta 17m 10s

## 3. Check the data matches the paper (7,909 images, 82 patients: 24 benign, 58 malignant)

In [ ]:
from breakhis.data import build_metadata, summarize
df = build_metadata(DATA_DIR)
print(len(df), 'images')
summarize(df)

## 4. Run the experiment
Quick first run: one magnification (200x), 3 seeds, 5 epochs → about 10–20 minutes on a T4.
Both split types use the **same model and settings**; only the split changes.

In [ ]:
!python -m breakhis.experiments.leakage --data_dir "$DATA_DIR" --out_dir "$RESULTS/leakage_200x" \
    --magnification 200 --seeds 0 1 2 --epochs 5

## 5. Results

In [ ]:
import pandas as pd
res = pd.read_csv(f'{RESULTS}/leakage_200x/leakage_results.csv')
cols = ['split','seed','patients_in_both','accuracy','balanced_accuracy','bal_acc_ci_low','bal_acc_ci_high','auroc','patient_balanced_accuracy','n_test_patients']
display(res[cols].round(3))
display(res.groupby('split')[['accuracy','balanced_accuracy','auroc','patient_balanced_accuracy']].agg(['mean','std']).round(3))

## 6. What to send back to Claude
Copy the two tables above into the chat. We'll read them together:
- How big is the gap between `image` and `patient` splits?
- How wide are the confidence intervals (only ~16 test patients per fold)?
- Does patient-level accuracy differ from image-level?